<a href="https://colab.research.google.com/github/rajeshwar-sopho/collab-notebooks/blob/main/ai_engineering/ai_judge_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Judge Lab
This notebook covers three types of AI judges (Reward Model, Reference-Based, Preference Model) using MLflow, RAGAS, and LlamaIndex.

**Requirements:** A Gemini API key for the evaluator LLM.

## 1. Setup & Install Dependencies

In [21]:
!pip install -q jedi jsonref google-genai
!pip install -q llama-index-llms-google-genai llama-index-embeddings-google-genai
!pip install -q requests==2.32.4
!pip install -q transformers torch google-generativeai "ragas" datasets \
    llama-index llama-index-llms-gemini llama-index-embeddings-gemini \
    mlflow litellm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.1 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.


In [14]:
import os
from getpass import getpass

GEMINI_API_KEY = getpass("Enter your Gemini API key: ")
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

Enter your Gemini API key: ··········


## 2. Load the Response-Generation Model
We use Gemini as our primary AI model to generate responses that will later be evaluated by different judges.

In [8]:
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)
# At the time of creation this was the cheapest model available from google update it if this is not available anymore.
generator_model = genai.GenerativeModel("gemini-2.5-flash-lite")

def generate_response(prompt: str) -> str:
    """Generate a response using Gemini."""
    response = generator_model.generate_content(prompt)
    return response.text

# Test
test_prompt = "Explain what photosynthesis is in 2-3 sentences."
test_response = generate_response(test_prompt)
print(f"Prompt: {test_prompt}")
print(f"Response: {test_response}")

Prompt: Explain what photosynthesis is in 2-3 sentences.
Response: Photosynthesis is the process plants use to convert light energy into chemical energy. They use sunlight, water, and carbon dioxide to create glucose (sugar) for food and release oxygen as a byproduct. This essential process forms the base of most food chains on Earth.


## 3. Prepare Evaluation Dataset
A shared set of prompts, generated responses, reference answers, and contexts used across all judges.

In [9]:
eval_questions = [
    "What is the capital of France?",
    "Explain how neural networks learn.",
    "What causes rainfall?",
    "What is the speed of light?",
]

reference_answers = [
    "The capital of France is Paris.",
    "Neural networks learn by adjusting weights through backpropagation, minimizing the loss function over training data.",
    "Rainfall is caused by the condensation of water vapor in the atmosphere forming clouds, which release precipitation when droplets grow heavy enough.",
    "The speed of light in vacuum is approximately 299,792,458 meters per second.",
]

contexts = [
    ["France is a country in Western Europe. Paris is its capital and largest city."],
    ["Neural networks consist of layers of interconnected nodes. They learn by adjusting connection weights using backpropagation and gradient descent."],
    ["Water evaporates from surfaces, rises into the atmosphere, cools and condenses into clouds. When droplets grow large enough, they fall as rain."],
    ["Light travels at a constant speed in vacuum. This speed is approximately 3 x 10^8 meters per second."],
]

# Generate responses for each question
generated_responses = [generate_response(q) for q in eval_questions]

for i, (q, r) in enumerate(zip(eval_questions, generated_responses)):
    print(f"Q{i+1}: {q}")
    print(f"A{i+1}: {r[:120]}...\n")

Q1: What is the capital of France?
A1: The capital of France is **Paris**....

Q2: Explain how neural networks learn.
A2: Neural networks learn through a process of **iterative adjustment** of their internal parameters, primarily **weights** ...

Q3: What causes rainfall?
A3: Rainfall is a fundamental part of the Earth's water cycle and is caused by a complex interplay of atmospheric processes....

Q4: What is the speed of light?
A4: The speed of light in a vacuum is a fundamental physical constant, denoted by the letter **c**. Its exact value is **299...



---
## Part A: Reward Model Judge (Cappy)
Cappy is a 360M parameter model that scores response correctness from 0 to 1. It acts as a lightweight reward model — no LLM judge needed.

In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

cappy_tokenizer = AutoTokenizer.from_pretrained("btan2/cappy-large")
cappy_model = AutoModelForSequenceClassification.from_pretrained("btan2/cappy-large")
cappy_model.eval()

def cappy_score(instruction: str, response: str) -> float:
    """Score a response using Cappy reward model. Returns 0-1."""
    inputs = cappy_tokenizer([(instruction, response)], return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        score = cappy_model(**inputs).logits[0][0].item()
    return score

print("=== Cappy Reward Model Scores ===")
for q, resp in zip(eval_questions, generated_responses):
    score = cappy_score(q, resp)
    print(f"Q: {q}")
    print(f"Score: {score:.4f}\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/738 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

=== Cappy Reward Model Scores ===
Q: What is the capital of France?
Score: 0.5946

Q: Explain how neural networks learn.
Score: 1.0099

Q: What causes rainfall?
Score: 0.9277

Q: What is the speed of light?
Score: 0.9995



In [11]:
# Compare good vs bad responses with Cappy
prompt = "What is the capital of Germany?"
good_response = "The capital of Germany is Berlin."
bad_response = "I like pizza and cats are fluffy."

score_good = cappy_score(prompt, good_response)
score_bad = cappy_score(prompt, bad_response)

print(f"Prompt: {prompt}")
print(f"Good response score: {score_good:.4f}")
print(f"Bad response score:  {score_bad:.4f}")
print(f"Cappy correctly prefers the good response: {score_good > score_bad}")

Prompt: What is the capital of Germany?
Good response score: 0.4387
Bad response score:  0.0028
Cappy correctly prefers the good response: True


---
## Part B: Reference-Based Judge
Evaluates responses by comparing against ground-truth references using LLM judges.

### B1. Reference-Based Judge with RAGAS
Uses Gemini as the evaluator LLM with Faithfulness and AnswerCorrectness metrics.

In [19]:
from datasets import Dataset
from ragas import evaluate
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.metrics import AnswerCorrectness, Faithfulness

# Install the new google-genai SDK if not already present
# !pip install -q google-genai

from google import genai

# Initialize Gemini client (new SDK)
genai_client = genai.Client(api_key=GEMINI_API_KEY)

# Create LLM and Embeddings both using Gemini
evaluator_llm = llm_factory("gemini-2.5-flash-lite", provider="google", client=genai_client)
evaluator_embeddings = embedding_factory("google", client=genai_client)

# Build evaluation dataset
ragas_data = {
    "question": eval_questions,
    "answer": generated_responses,
    "contexts": contexts,
    "ground_truth": reference_answers,
}
ragas_dataset = Dataset.from_dict(ragas_data)

# Pass embeddings explicitly to AnswerCorrectness
ragas_metrics = [
    Faithfulness(llm=evaluator_llm),
    AnswerCorrectness(llm=evaluator_llm, embeddings=evaluator_embeddings),
]

ragas_results = evaluate(ragas_dataset, metrics=ragas_metrics)
print("=== RAGAS Reference-Based Results ===")
print(ragas_results)
ragas_results.to_pandas()

/tmp/ipykernel_2218/3218001786.py:5: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import AnswerCorrectness, Faithfulness
/tmp/ipykernel_2218/3218001786.py:5: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import AnswerCorrectness, Faithfulness
/tmp/ipykernel_2218/3218001786.py:17: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = embedding_factory("google", client=genai_client)


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

=== RAGAS Reference-Based Results ===
{'faithfulness': 0.3603, 'answer_correctness': 0.6158}


,user_input,retrieved_contexts,response,reference,faithfulness,answer_correctness
0,What is the capital of France?,[France is a country in Western Europe. Paris ...,The capital of France is **Paris**.,The capital of France is Paris.,1.000000,0.969788
1,Explain how neural networks learn.,[Neural networks consist of layers of intercon...,Neural networks learn through a process of **i...,Neural networks learn by adjusting weights thr...,0.150000,0.320233
2,What causes rainfall?,"[Water evaporates from surfaces, rises into th...",Rainfall is a fundamental part of the Earth's ...,Rainfall is caused by the condensation of wate...,0.068966,0.713067
3,What is the speed of light?,[Light travels at a constant speed in vacuum. ...,The speed of light in a vacuum is a fundamenta...,The speed of light in vacuum is approximately ...,0.222222,0.460151


### B2. Reference-Based Judge with LlamaIndex
Uses CorrectnessEvaluator and FaithfulnessEvaluator with Gemini as the judge LLM.

In [22]:
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.evaluation import CorrectnessEvaluator, FaithfulnessEvaluator

llm_judge = GoogleGenAI(
    model="gemini-2.5-flash-lite",
    api_key=GEMINI_API_KEY,
)

correctness_evaluator = CorrectnessEvaluator(llm=llm_judge)
faithfulness_evaluator = FaithfulnessEvaluator(llm=llm_judge)

print("=== LlamaIndex Reference-Based Results ===")
for i in range(len(eval_questions)):
    correctness_result = correctness_evaluator.evaluate(
        query=eval_questions[i],
        response=generated_responses[i],
        reference=reference_answers[i],
    )
    faithfulness_result = faithfulness_evaluator.evaluate(
        query=eval_questions[i],
        response=generated_responses[i],
        contexts=contexts[i],
    )
    print(f"Q: {eval_questions[i]}")
    print(f"  Correctness score: {correctness_result.score}, Passing: {correctness_result.passing}")
    print(f"  Faithfulness: {faithfulness_result.passing}")
    print(f"  Feedback: {correctness_result.feedback[:150]}...\n")

=== LlamaIndex Reference-Based Results ===
Q: What is the capital of France?
  Correctness score: 5.0, Passing: True
  Faithfulness: True
  Feedback: The generated answer is perfectly relevant and correct. It directly answers the user's question with the correct information....

Q: Explain how neural networks learn.
  Correctness score: 5.0, Passing: True
  Faithfulness: True
  Feedback: The generated answer is excellent. It provides a comprehensive and easy-to-understand explanation of how neural networks learn, covering all the key c...

Q: What causes rainfall?
  Correctness score: 5.0, Passing: True
  Faithfulness: True
  Feedback: The generated answer is highly relevant and provides a comprehensive and accurate explanation of what causes rainfall, going into detail about the var...

Q: What is the speed of light?
  Correctness score: 5.0, Passing: True
  Faithfulness: True
  Feedback: The generated answer is excellent. It provides the exact value of the speed of light in meters pe

### B3. Reference-Based Judge with MLflow
Uses MLflow's built-in Correctness scorer with Gemini as the judge model.

In [23]:
import mlflow
from mlflow.genai.scorers import Correctness, Guidelines

# Build MLflow evaluation dataset
mlflow_eval_data = [
    {
        "inputs": {"question": q},
        "outputs": {"response": r},
        "expectations": {"expected_response": ref},
    }
    for q, r, ref in zip(eval_questions, generated_responses, reference_answers)
]

# Use Gemini as the judge via litellm integration
mlflow_results = mlflow.genai.evaluate(
    data=mlflow_eval_data,
    scorers=[
        Correctness(model="gemini-2.5-flash-lite"),
        Guidelines(
            name="conciseness",
            guidelines="The response should be concise and directly answer the question without unnecessary filler.",
            model="gemini-2.5-flash-lite",
        ),
    ],
)

print("=== MLflow Reference-Based Results ===")
print(mlflow_results.metrics)

2026/03/29 09:03:40 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/29 09:03:40 INFO mlflow.store.db.utils: Updating database tables
2026/03/29 09:03:42 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.


Evaluating:   0%|          | 0/4 [Elapsed: 00:00, Remaining: ?] 


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: exultant-fowl-783
  Run ID: 39fe1f70e1b84eeba0bdc5803a526b04

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.

=== MLflow Reference-Based Results ===
{}


---
## Part C: Preference Model Judge
Compares two candidate responses and picks the better one. This is the pairwise comparison approach.

### C1. Preference Judge with Cappy (Reward-Based Pairwise)
Score both candidates independently and prefer the higher-scoring one.

In [41]:
# Generate two candidate responses per question with different temperatures
candidate_a_responses = generated_responses  # already generated
candidate_b_responses = [
    generate_response(f"Answer very briefly: {q}") for q in eval_questions
]

print("=== Cappy Preference Comparison ===")
for i, q in enumerate(eval_questions):
    score_a = cappy_score(q, candidate_a_responses[i])
    score_b = cappy_score(q, candidate_b_responses[i])
    preferred = "A" if score_a > score_b else "B"
    print(f"Q: {q}")
    print(f"  Candidate A score: {score_a:.4f}")
    print(f"  Candidate B score: {score_b:.4f}")
    print(f"  Preferred: Candidate {preferred}\n")

=== Cappy Preference Comparison ===
Q: What is the capital of France?
  Candidate A score: 0.5946
  Candidate B score: 0.7453
  Preferred: Candidate B

Q: Explain how neural networks learn.
  Candidate A score: 1.0099
  Candidate B score: 0.9830
  Preferred: Candidate A

Q: What causes rainfall?
  Candidate A score: 0.9277
  Candidate B score: 0.4905
  Preferred: Candidate A

Q: What is the speed of light?
  Candidate A score: 0.9995
  Candidate B score: 0.6974
  Preferred: Candidate A



### C2. LLM-Based Preference Judge with Gemini
Uses Gemini to directly compare two responses and explain which is better.

In [42]:
PREFERENCE_PROMPT = """You are a response quality judge. Given a question and two candidate responses, decide which response is better.

Question: {question}

Response A: {response_a}

Response B: {response_b}

Evaluate on: accuracy, completeness, and clarity.
Output format:
Winner: [A or B]
Reason: [1-2 sentence explanation]"""

def preference_judge(question: str, response_a: str, response_b: str) -> str:
    """Use Gemini as a preference judge between two responses."""
    prompt = PREFERENCE_PROMPT.format(
        question=question, response_a=response_a, response_b=response_b
    )
    return generate_response(prompt)

print("=== Gemini Preference Judge ===")
for i, q in enumerate(eval_questions):
    result = preference_judge(q, candidate_a_responses[i], candidate_b_responses[i])
    print(f"Q: {q}")
    print(f"Judgment: {result}\n")

=== Gemini Preference Judge ===
Q: What is the capital of France?
Judgment: Winner: A
Reason: Response A is more complete and clear by explicitly stating "The capital of France is Paris." Response B is accurate but lacks the full context and clarity of the question.

Q: Explain how neural networks learn.
Judgment: Winner: A
Reason: Response A provides a comprehensive and well-structured explanation of how neural networks learn, covering the underlying mechanisms, the learning process, and key concepts with helpful analogies. Response B is very brief and lacks the detail needed for a thorough understanding.

Q: What causes rainfall?
Judgment: Winner: A
Reason: Response A provides a comprehensive and detailed explanation of the causes of rainfall, breaking down the process into logical steps. Response B is too brief and lacks the necessary detail to fully answer the question.

Q: What is the speed of light?
Judgment: Winner: A
Reason: Response A provides the exact and commonly used appro

### C3. Preference Judge with MLflow Custom Scorer

In [45]:
from mlflow.genai.judges import make_judge

preference_mlflow_judge = make_judge(
    name="pairwise_preference",
    instructions=(
        "Compare response_a and response_b in {{ outputs }} for the question in {{ inputs }}. "
        "Determine which response is more accurate, complete, and clear. "
        "Return 'A' if response_a is better, 'B' if response_b is better."
    ),
    feedback_value_type=str,
    model="gemini:/gemini-2.5-flash-lite",
)

# Build pairwise dataset
pairwise_data = [
    {
        "inputs": {"question": q},
        "outputs": {"response_a": a, "response_b": b},
    }
    for q, a, b in zip(eval_questions, candidate_a_responses, candidate_b_responses)
]

pairwise_results = mlflow.genai.evaluate(
    data=pairwise_data,
    scorers=[preference_mlflow_judge],
)

print("=== MLflow Preference Judge Results ===")
print(pairwise_results.metrics)

Evaluating:   0%|          | 0/4 [Elapsed: 00:00, Remaining: ?] 


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: adventurous-cod-677
  Run ID: 5e9549cd1b274f68b4ea26b2567b6fbf

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.

=== MLflow Preference Judge Results ===
{}


### C4. Preference Judge with LlamaIndex PairwiseComparisonEvaluator

In [46]:
from llama_index.core.evaluation import PairwiseComparisonEvaluator

pairwise_evaluator = PairwiseComparisonEvaluator(llm=llm_judge)

print("=== LlamaIndex Pairwise Preference Results ===")
for i, q in enumerate(eval_questions):
    result = pairwise_evaluator.evaluate(
        query=q,
        response=candidate_a_responses[i],
        second_response=candidate_b_responses[i],
    )
    print(f"Q: {q}")
    print(f"  Winner: {'A' if result.score == 1 else 'B'}")
    print(f"  Feedback: {result.feedback[:150]}...\n")

=== LlamaIndex Pairwise Preference Results ===
Q: What is the capital of France?
  Winner: A
  Feedback: Both assistants correctly identify Paris as the capital of France. Assistant A provides a more complete sentence, which is generally preferred for cla...

Q: Explain how neural networks learn.
  Winner: A
  Feedback: Assistant A provides a comprehensive and detailed explanation of how neural networks learn, breaking down the process into logical steps and using ana...

Q: What causes rainfall?
  Winner: A
  Feedback: Assistant A provides a comprehensive and detailed explanation of the process of rainfall, breaking it down into logical steps from evaporation to prec...

Q: What is the speed of light?
  Winner: A
  Feedback: Assistant A provides a more comprehensive answer by giving the exact value of the speed of light in meters per second, as well as common approximation...



---
## Summary

| Judge Type | Tool | Key Idea |
|---|---|---|
| **Reward Model** | Cappy (btan2/cappy-large) | Small classifier outputs 0-1 score per response, no LLM call needed |
| **Reference-Based** | RAGAS, LlamaIndex, MLflow | Compares response against ground-truth reference using LLM judge |
| **Preference Model** | Cappy pairwise, Gemini judge, MLflow, LlamaIndex | Compares two candidate responses and picks the better one |